# xarray + THREDDS: Keycloak-authenticated OPeNDAP access

Demonstrates an auth pattern used for authentication against the `NILU` THREDDS server reverse proxy solution, with a standalone script in `thredds_auth.py`. This notebook walsk through the following steps:

1. Log in to Keycloak (Resource-Owner-Password-Credentials grant) to get an `access_token` + `refresh_token`.
2. Open a **public** OPeNDAP dataset with plain `xarray` (`engine="netcdf4"`), no auth needed.
3. Open a **restricted** dataset by attaching the access token as a `Bearer` header on the `requests.Session` xarray uses (`engine="pydap"`) — `netCDF4`'s own DAP client can't send custom headers, `pydap` can.
4. Show a restricted dataset failing fast (not hanging) when no/invalid token is supplied.
5. Refresh the access token (~5 minute lifetime) and keep working without re-entering credentials.

In [1]:
import base64
import json
import time
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

import thredds_auth

## 1. Log in

Credentials are prompted interactively (never written to disk). Set `EBAS_USERNAME` /
`EBAS_PASSWORD` in `.env` instead if you need a non-interactive run.

In [2]:
import os

username = os.getenv("EBAS_USERNAME") or input("Username: ")
password = os.getenv("EBAS_PASSWORD") or getpass("Password: ")

state = thredds_auth.login(username, password)
print(f"access_token expires at {time.ctime(state.expires_at)}")

Username:  jhol
Password:  ········


access_token expires at Fri Sep 11 06:54:41 2026


In [3]:
def decode_jwt_payload(token: str) -> dict:
    payload_b64 = token.split(".")[1]
    padded = payload_b64 + "=" * (-len(payload_b64) % 4)
    return json.loads(base64.urlsafe_b64decode(padded))


claims = decode_jwt_payload(state.access_token)
print(f"issued at:  {time.ctime(claims['iat'])}")
print(f"expires at: {time.ctime(claims['exp'])}")
print(f"lifetime:   {claims['exp'] - claims['iat']} s")

issued at:  Fri Sep 11 06:49:41 2026
expires at: Fri Sep 11 06:54:41 2026
lifetime:   300 s


## 2. Public dataset — no auth needed

Plain `xarray.open_dataset(url, engine="netcdf4")`, `token=None`.

In [4]:
PUBLIC_OPENDAP_URL = "https://thredds.nilu.no/thredds/dodsC/ebas_doi/6E/DB/9H/6EDB-9HTZ.nc"

ds_public = thredds_auth.open_dataset(PUBLIC_OPENDAP_URL)
ds_public

<xarray.Dataset> Size: 2kB
Dimensions:             (time: 12, tbnds: 2, metadata_time: 1,
                         qc_TBBPA_flags: 4, qc_a_HBCD_flags: 2,
                         qc_b_HBCD_flags: 2, qc_g_HBCD_flags: 3)
Coordinates:
  * time                (time) datetime64[ns] 96B 2025-01-16T05:18:00 ... 202...
  * metadata_time       (metadata_time) datetime64[ns] 8B 2025-06-29T17:09:00
Dimensions without coordinates: tbnds, qc_TBBPA_flags, qc_a_HBCD_flags,
                                qc_b_HBCD_flags, qc_g_HBCD_flags
Data variables: (12/14)
    time_bnds           (time, tbnds) datetime64[ns] 192B ...
    metadata_time_bnds  (metadata_time, tbnds) datetime64[ns] 16B ...
    v_TBBPA             (time) float64 96B ...
    qc_TBBPA            (qc_TBBPA_flags, time) float64 384B ...
    meta_TBBPA          (metadata_time) |S64 64B ...
    v_a_HBCD            (time) float64 96B ...
    ...                  ...
    v_b_HBCD            (time) float64 96B ...
    qc_b_HBCD           (qc_b_HBCD_flags, time) float64 192B ...
    meta_b_HBCD         (metadata_time) |S64 64B ...
    v_g_HBCD            (time) float64 96B ...
    qc_g_HBCD           (qc_g_HBCD_flags, time) float64 288B ...
    meta_g_HBCD         (metadata_time) |S64 64B ...
Attributes: (12/112)
    Conventions:                              CF-1.8, ACDD-1.3
    featureType:                              timeSeries
    title:                                    POPs (persistent organic pollut...
    summary:                                  POPs (persistent organic pollut...
    keywords:                                 "Birkenes II" NO0002R TBBPA a_H...
    id:                                       6EDB-9HTZ.nc
    ...                                       ...
    Metadata_Conventions:                     Unidata Dataset Discovery v1.0
    geospatial_lat_units:                     degrees_north
    geospatial_lon_units:                     degrees_east
    standard_name_vocabulary:                 CF-1.7, ACDD-1.3
    history:                                  None
    creator_url:                              ebas.nilu.no

## 3. Restricted dataset — with the access token

Same call, passing `token=state.access_token`. Internally this switches to
`engine="pydap"` with a `requests.Session` carrying `Authorization: Bearer <token>`.

In [5]:
RESTRICTED_OPENDAP_URL = "https://thredds.nilu.no/thredds/dodsC/ebas_doi/XK/QB/YA/XKQB-YAPG.nc"

ds_restricted = thredds_auth.open_dataset(RESTRICTED_OPENDAP_URL, token=state.access_token)
ds_restricted

<xarray.Dataset> Size: 400kB
Dimensions:               (time: 1827, tbnds: 2, metadata_time: 5,
                           qc_3_methylbutanal_flags: 2,
                           qc_benzaldehyde_flags: 2, qc_ethanal_flags: 2,
                           qc_methanal_flags: 2, qc_n_butanal_flags: 2,
                           qc_pentanal_flags: 2, qc_propanal_flags: 2,
                           qc_propanone_flags: 2)
Coordinates:
  * time                  (time) datetime64[ns] 15kB 2020-01-01T12:00:00 ... ...
  * metadata_time         (metadata_time) datetime64[ns] 40B 2020-07-02 ... 2...
Dimensions without coordinates: tbnds, qc_3_methylbutanal_flags,
                                qc_benzaldehyde_flags, qc_ethanal_flags,
                                qc_methanal_flags, qc_n_butanal_flags,
                                qc_pentanal_flags, qc_propanal_flags,
                                qc_propanone_flags
Data variables: (12/26)
    time_bnds             (time, tbnds) datetime64[ns] 29kB ...
    metadata_time_bnds    (metadata_time, tbnds) datetime64[ns] 80B ...
    v_3_methylbutanal     (time) float64 15kB ...
    qc_3_methylbutanal    (qc_3_methylbutanal_flags, time) float64 29kB ...
    meta_3_methylbutanal  (metadata_time) |S128 640B ...
    v_benzaldehyde        (time) float64 15kB ...
    ...                    ...
    v_propanal            (time) float64 15kB ...
    qc_propanal           (qc_propanal_flags, time) float64 29kB ...
    meta_propanal         (metadata_time) |S128 640B ...
    v_propanone           (time) float64 15kB ...
    qc_propanone          (qc_propanone_flags, time) float64 29kB ...
    meta_propanone        (metadata_time) |S128 640B ...
Attributes: (12/87)
    Conventions:                   CF-1.8, ACDD-1.3
    featureType:                   timeSeries
    title:                         VOC (OVOCs) at Lahemaa
    summary:                       VOC (OVOCs) at Lahemaa. These measurements...
    keywords:                      Lahemaa EE0009R 3-methylbutanal benzaldehy...
    id:                            XKQB-YAPG.nc
    ...                            ...
    Metadata_Conventions:          Unidata Dataset Discovery v1.0
    geospatial_lat_units:          degrees_north
    geospatial_lon_units:          degrees_east
    standard_name_vocabulary:      CF-1.7, ACDD-1.3
    history:                       None
    creator_url:                   ebas.nilu.no

## 4. Restricted dataset — without a valid token

`open_dataset()` probes `{url}.dods?time[0:1:0]` before handing off to xarray, so a
401/403 is raised as `DatasetAccessError` in milliseconds instead of `netCDF4` silently
retrying for 40-120s.

In [6]:
try:
    thredds_auth.open_dataset(RESTRICTED_OPENDAP_URL, token=None)
except thredds_auth.DatasetAccessError as exc:
    print(f"failed fast as expected: HTTP {exc.status_code} — {exc}")

failed fast as expected: HTTP 401 — Dataset is restricted


## 5. Refresh tokens

Access tokens are short-lived (~5 minutes, see the `iat`/`exp` claims above).
`ensure_fresh()` is the session-less equivalent of production's
`keycloak_auth.ensure_fresh_access_token()`: it refreshes proactively whenever the
access token is within `margin_s` of expiring, using the `refresh_token` — no password
re-entry needed. Call it before any THREDDS request in a long-running notebook/script.

In [9]:
old_access_token = state.access_token

# Force ensure_fresh() to refresh right now, regardless of actual expiry,
# by pretending the token is already past the margin.
state.expires_at = time.time()
state = thredds_auth.ensure_fresh(state)

assert state.access_token != old_access_token, "expected a new access token"
print(f"refreshed — new access_token expires at {time.ctime(state.expires_at)}")

# Same restricted dataset still opens with the refreshed token.
thredds_auth.open_dataset(RESTRICTED_OPENDAP_URL, token=state.access_token)

refreshed — new access_token expires at Fri Sep 11 07:06:26 2026


<xarray.Dataset> Size: 400kB
Dimensions:               (time: 1827, tbnds: 2, metadata_time: 5,
                           qc_3_methylbutanal_flags: 2,
                           qc_benzaldehyde_flags: 2, qc_ethanal_flags: 2,
                           qc_methanal_flags: 2, qc_n_butanal_flags: 2,
                           qc_pentanal_flags: 2, qc_propanal_flags: 2,
                           qc_propanone_flags: 2)
Coordinates:
  * time                  (time) datetime64[ns] 15kB 2020-01-01T12:00:00 ... ...
  * metadata_time         (metadata_time) datetime64[ns] 40B 2020-07-02 ... 2...
Dimensions without coordinates: tbnds, qc_3_methylbutanal_flags,
                                qc_benzaldehyde_flags, qc_ethanal_flags,
                                qc_methanal_flags, qc_n_butanal_flags,
                                qc_pentanal_flags, qc_propanal_flags,
                                qc_propanone_flags
Data variables: (12/26)
    time_bnds             (time, tbnds) datetime64[ns] 29kB ...
    metadata_time_bnds    (metadata_time, tbnds) datetime64[ns] 80B ...
    v_3_methylbutanal     (time) float64 15kB ...
    qc_3_methylbutanal    (qc_3_methylbutanal_flags, time) float64 29kB ...
    meta_3_methylbutanal  (metadata_time) |S128 640B ...
    v_benzaldehyde        (time) float64 15kB ...
    ...                    ...
    v_propanal            (time) float64 15kB ...
    qc_propanal           (qc_propanal_flags, time) float64 29kB ...
    meta_propanal         (metadata_time) |S128 640B ...
    v_propanone           (time) float64 15kB ...
    qc_propanone          (qc_propanone_flags, time) float64 29kB ...
    meta_propanone        (metadata_time) |S128 640B ...
Attributes: (12/87)
    Conventions:                   CF-1.8, ACDD-1.3
    featureType:                   timeSeries
    title:                         VOC (OVOCs) at Lahemaa
    summary:                       VOC (OVOCs) at Lahemaa. These measurements...
    keywords:                      Lahemaa EE0009R 3-methylbutanal benzaldehy...
    id:                            XKQB-YAPG.nc
    ...                            ...
    Metadata_Conventions:          Unidata Dataset Discovery v1.0
    geospatial_lat_units:          degrees_north
    geospatial_lon_units:          degrees_east
    standard_name_vocabulary:      CF-1.7, ACDD-1.3
    history:                       None
    creator_url:                   ebas.nilu.no

### Caveats not covered above

- The `refresh_token` itself also expires (typically a longer-lived Keycloak SSO
  session setting) — once it does, `refresh()` fails and `login()` must be called again.
    - At the time of writing this is set to 30 minute idle, and a re-login is required every 10 hours.
- This demo uses the direct password grant (`ebas-cli`, a public client configured for
  it) for simplicity. A browser-based app would use an authorization-code + PKCE flow
  instead — the token handling downstream (`Bearer` header, `ensure_fresh` pattern) is
  identical either way.